# Black Friday Sales: fatores que influenciam compras premium

Este notebook extrai o dataset público [Retail Black Friday Sales Dataset](https://www.kaggle.com/datasets/noopurbhatt/retail-black-friday-sales-dataset), realiza uma análise exploratória e constrói modelos de Machine Learning para responder:

> **Quais fatores influenciam a probabilidade de um cliente realizar compras premium durante a Black Friday?**

A análise trabalha com:

- **Feature Importance** em modelos supervisionados;
- **SHAP** para explicabilidade global e local;
- **Explainable AI** para transformar o modelo em evidências interpretáveis.

## Definição operacional de compra premium

Como o dataset não traz uma coluna explícita indicando compra premium, este notebook define uma transação premium como aquela cujo valor de `Purchase` está no percentil 75 ou acima. Ou seja, compras no quartil superior de gasto são tratadas como premium.

Essa definição pode ser alterada na variável `PREMIUM_QUANTILE`.

## 1. Preparação do ambiente

Execute a próxima célula se o ambiente ainda não tiver as bibliotecas necessárias. Ela instala pacotes usados para extração, análise, modelagem e explicabilidade.

In [1]:
#%pip install -U kagglehub pandas numpy matplotlib seaborn scikit-learn shap joblib

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="viridis")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

RANDOM_STATE = 42
PREMIUM_QUANTILE = 0.75
DATASET_SLUG = "noopurbhatt/retail-black-friday-sales-dataset"
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

print(f"Diretório do projeto: {PROJECT_DIR}")

Diretório do projeto: /Users/evandrocoelho/Library/CloudStorage/OneDrive-LeadsCard/Pessoal/educacao/puc/_tcc/black-friday-sales


## 2. Extração dos dados do Kaggle

A próxima célula baixa o dataset usando `kagglehub`. Para datasets públicos, o pacote costuma resolver o download diretamente. Se houver erro de autenticação, configure um token Kaggle em `~/.kaggle/kaggle.json` ou baixe o CSV manualmente e coloque-o em `data/`.

O notebook procura automaticamente arquivos `.csv` baixados ou salvos localmente.

In [3]:
def download_dataset(dataset_slug: str = DATASET_SLUG) -> Path:
    """Baixa o dataset via kagglehub e retorna o diretório local."""
    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError(
            "Instale o kagglehub com `%pip install kagglehub` ou baixe o CSV manualmente para data/."
        ) from exc

    dataset_path = Path(kagglehub.dataset_download(dataset_slug))
    print(f"Dataset baixado/cacheado em: {dataset_path}")
    return dataset_path


def find_csv_files(*directories: Path) -> list[Path]:
    csv_files = []
    for directory in directories:
        if directory.exists():
            csv_files.extend(sorted(directory.rglob("*.csv")))
    return csv_files

try:
    downloaded_dir = download_dataset()
except Exception as exc:
    downloaded_dir = None
    print("Não foi possível baixar automaticamente pelo Kaggle.")
    print(f"Motivo: {exc}")
    print("Fallback: coloque o arquivo CSV do Kaggle dentro da pasta data/ e reexecute as próximas células.")

candidate_dirs = [DATA_DIR]
if downloaded_dir is not None:
    candidate_dirs.insert(0, downloaded_dir)

csv_files = find_csv_files(*candidate_dirs)
if not csv_files:
    raise FileNotFoundError("Nenhum arquivo CSV encontrado. Baixe o dataset e coloque o CSV em data/.")

csv_files

Não foi possível baixar automaticamente pelo Kaggle.
Motivo: Instale o kagglehub com `%pip install kagglehub` ou baixe o CSV manualmente para data/.
Fallback: coloque o arquivo CSV do Kaggle dentro da pasta data/ e reexecute as próximas células.


FileNotFoundError: Nenhum arquivo CSV encontrado. Baixe o dataset e coloque o CSV em data/.

In [4]:
# Seleciona o maior CSV encontrado, uma heurística útil caso o pacote baixe arquivos auxiliares.
csv_path = max(csv_files, key=lambda path: path.stat().st_size)
print(f"Arquivo selecionado: {csv_path}")

df_raw = pd.read_csv(csv_path)
df_raw.head()

ValueError: max() iterable argument is empty

## 3. Entendimento inicial e qualidade dos dados

Nesta etapa verificamos formato, nulos, tipos de variáveis e estatísticas descritivas. Isso orienta tanto a EDA quanto o pré-processamento do modelo.

In [ ]:
df = df_raw.copy()
df.columns = [column.strip() for column in df.columns]

required_columns = {
    "User_ID",
    "Product_ID",
    "Gender",
    "Age",
    "Occupation",
    "City_Category",
    "Stay_In_Current_City_Years",
    "Marital_Status",
    "Product_Category_1",
    "Purchase",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Colunas esperadas ausentes no dataset: {sorted(missing_columns)}")

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]:,}")
display(df.info())
display(df.describe(include="all").T)

In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_pct=lambda data: data["missing_count"] / len(df))
    .sort_values("missing_count", ascending=False)
)

missing_summary[missing_summary["missing_count"] > 0]

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()

print("Variáveis numéricas:", numeric_columns)
print("Variáveis categóricas:", categorical_columns)

for column in categorical_columns:
    print(f"\n{column}: {df[column].nunique()} categorias")
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

## 4. Análise exploratória

A EDA procura sinais iniciais de associação entre perfil do cliente, características do produto e valor de compra. Essas relações serão testadas depois por modelos supervisionados.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Purchase"], bins=50, kde=True, ax=axes[0])
axes[0].axvline(df["Purchase"].quantile(PREMIUM_QUANTILE), color="red", linestyle="--", label="limiar premium")
axes[0].set_title("Distribuição do valor de compra")
axes[0].legend()

sns.boxplot(x=df["Purchase"], ax=axes[1])
axes[1].set_title("Boxplot do valor de compra")

plt.tight_layout()
plt.show()

purchase_quantiles = df["Purchase"].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
purchase_quantiles.to_frame("Purchase")

In [ ]:
df["is_premium_purchase"] = (df["Purchase"] >= df["Purchase"].quantile(PREMIUM_QUANTILE)).astype(int)
premium_threshold = df["Purchase"].quantile(PREMIUM_QUANTILE)

print(f"Limiar de compra premium (P{int(PREMIUM_QUANTILE * 100)}): {premium_threshold:,.2f}")
print(f"Taxa de compras premium: {df['is_premium_purchase'].mean():.2%}")

df[["Purchase", "is_premium_purchase"]].head()

In [ ]:
def plot_premium_rate_by(column: str, top_n: int | None = None) -> pd.DataFrame:
    summary = (
        df.groupby(column, dropna=False)
        .agg(
            premium_rate=("is_premium_purchase", "mean"),
            avg_purchase=("Purchase", "mean"),
            transactions=("Purchase", "size"),
        )
        .sort_values("premium_rate", ascending=False)
    )
    plot_data = summary.head(top_n) if top_n else summary

    plt.figure(figsize=(10, max(4, 0.35 * len(plot_data))))
    sns.barplot(data=plot_data.reset_index(), y=column, x="premium_rate", orient="h")
    plt.title(f"Taxa de compras premium por {column}")
    plt.xlabel("Taxa de compras premium")
    plt.ylabel(column)
    plt.xlim(0, min(1, plot_data["premium_rate"].max() * 1.15))
    plt.show()

    return summary

eda_dimensions = ["Gender", "Age", "Occupation", "City_Category", "Stay_In_Current_City_Years", "Marital_Status"]
for column in eda_dimensions:
    display(plot_premium_rate_by(column).head(20))

In [ ]:
product_category_columns = [column for column in df.columns if column.startswith("Product_Category")]

for column in product_category_columns:
    if column in df.columns:
        display(plot_premium_rate_by(column, top_n=20).head(20))

## 5. Preparação para Machine Learning

A tarefa será tratada como classificação binária:

- `1`: transação premium (`Purchase` no percentil 75 ou acima);
- `0`: transação não premium.

Importante: `Purchase` não entra como variável explicativa, porque foi usado para construir o alvo. `User_ID` e `Product_ID` também são removidos para reduzir memorização de identificadores. O foco fica em atributos interpretáveis de cliente, cidade e produto.

In [ ]:
target = "is_premium_purchase"
leakage_or_id_columns = ["Purchase", target, "User_ID", "Product_ID"]
feature_columns = [column for column in df.columns if column not in leakage_or_id_columns]

X = df[feature_columns].copy()
y = df[target].copy()

# Mantém categorias de produto como categóricas, apesar de estarem codificadas numericamente.
category_like_columns = [column for column in X.columns if column.startswith("Product_Category") or column == "Occupation"]
for column in category_like_columns:
    X[column] = X[column].astype("category")

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [column for column in X.columns if column not in numeric_features]

print("Features numéricas:", numeric_features)
print("Features categóricas:", categorical_features)
print("Distribuição do alvo:")
display(y.value_counts(normalize=True).rename("proportion").to_frame())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", encoder, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print(f"Treino: {X_train.shape}")
print(f"Teste: {X_test.shape}")

## 6. Modelagem

Serão comparados três modelos:

- Regressão Logística: baseline interpretável;
- Random Forest: modelo não linear com importância por impureza e permutação;
- HistGradientBoosting: modelo de boosting robusto para relações não lineares.

A métrica principal será **ROC AUC**, complementada por **Average Precision**, útil em problemas com classes desbalanceadas.

In [ ]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("preprocess", preprocessor),
            (
                "model",
                LogisticRegression(
                    max_iter=2_000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocess", preprocessor),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=250,
                    min_samples_leaf=20,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "HistGradientBoosting": Pipeline(
        steps=[
            ("preprocess", preprocessor),
            (
                "model",
                HistGradientBoostingClassifier(
                    learning_rate=0.08,
                    max_iter=250,
                    l2_regularization=0.01,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "accuracy": "accuracy",
}

cv_results = []
for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )
    cv_results.append(
        {
            "model": name,
            "roc_auc_mean": scores["test_roc_auc"].mean(),
            "roc_auc_std": scores["test_roc_auc"].std(),
            "avg_precision_mean": scores["test_average_precision"].mean(),
            "accuracy_mean": scores["test_accuracy"].mean(),
        }
    )

cv_results_df = pd.DataFrame(cv_results).sort_values("roc_auc_mean", ascending=False)
cv_results_df

In [ ]:
best_model_name = cv_results_df.iloc[0]["model"]
best_pipeline = models[best_model_name]
best_pipeline.fit(X_train, y_train)

if hasattr(best_pipeline.named_steps["model"], "predict_proba"):
    y_score = best_pipeline.predict_proba(X_test)[:, 1]
else:
    y_score = best_pipeline.decision_function(X_test)

y_pred = (y_score >= 0.5).astype(int)

print(f"Melhor modelo por validação cruzada: {best_model_name}")
print(f"ROC AUC teste: {roc_auc_score(y_test, y_score):.4f}")
print(f"Average Precision teste: {average_precision_score(y_test, y_score):.4f}")
print(f"Accuracy teste: {accuracy_score(y_test, y_pred):.4f}")
print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, target_names=["não premium", "premium"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
RocCurveDisplay.from_predictions(y_test, y_score, ax=axes[0])
axes[0].set_title(f"Curva ROC - {best_model_name}")

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1])
axes[1].set_title("Matriz de confusão")
axes[1].set_xlabel("Predito")
axes[1].set_ylabel("Real")
axes[1].set_xticklabels(["não premium", "premium"])
axes[1].set_yticklabels(["não premium", "premium"])

plt.tight_layout()
plt.show()

## 7. Feature Importance

A importância por permutação mede a queda de desempenho quando uma variável é embaralhada. Como ela é calculada sobre o pipeline completo, o resultado fica no nível das variáveis originais, o que facilita a interpretação de negócio.

In [ ]:
permutation = permutation_importance(
    best_pipeline,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

permutation_importance_df = (
    pd.DataFrame(
        {
            "feature": X_test.columns,
            "importance_mean": permutation.importances_mean,
            "importance_std": permutation.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(permutation_importance_df)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=permutation_importance_df.head(15),
    x="importance_mean",
    y="feature",
    xerr=permutation_importance_df.head(15)["importance_std"],
)
plt.title(f"Permutation Feature Importance - {best_model_name}")
plt.xlabel("Redução média no ROC AUC")
plt.ylabel("Variável")
plt.show()

In [ ]:
def get_transformed_feature_names(pipeline: Pipeline) -> list[str]:
    transformer = pipeline.named_steps["preprocess"]
    try:
        return transformer.get_feature_names_out().tolist()
    except Exception:
        names = []
        if numeric_features:
            names.extend(numeric_features)
        if categorical_features:
            encoder_step = transformer.named_transformers_["cat"]
            names.extend(encoder_step.get_feature_names_out(categorical_features).tolist())
        return names

transformed_feature_names = get_transformed_feature_names(best_pipeline)
model = best_pipeline.named_steps["model"]

if hasattr(model, "feature_importances_"):
    model_importance_df = (
        pd.DataFrame(
            {
                "feature": transformed_feature_names,
                "importance": model.feature_importances_,
            }
        )
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    display(model_importance_df.head(25))

    plt.figure(figsize=(10, 7))
    sns.barplot(data=model_importance_df.head(20), x="importance", y="feature")
    plt.title(f"Importância interna do modelo - {best_model_name}")
    plt.xlabel("Importância")
    plt.ylabel("Variável transformada")
    plt.show()
elif hasattr(model, "coef_"):
    coef_df = (
        pd.DataFrame(
            {
                "feature": transformed_feature_names,
                "coefficient": model.coef_[0],
                "abs_coefficient": np.abs(model.coef_[0]),
            }
        )
        .sort_values("abs_coefficient", ascending=False)
        .reset_index(drop=True)
    )
    display(coef_df.head(25))

    plt.figure(figsize=(10, 7))
    sns.barplot(data=coef_df.head(20), x="coefficient", y="feature")
    plt.axvline(0, color="black", linewidth=1)
    plt.title(f"Coeficientes mais relevantes - {best_model_name}")
    plt.xlabel("Coeficiente")
    plt.ylabel("Variável transformada")
    plt.show()
else:
    print("O modelo selecionado não expõe importância interna direta. Use a importância por permutação acima.")

## 8. SHAP e Explainable AI

SHAP estima quanto cada variável contribui para deslocar a previsão em direção à classe premium ou não premium. A análise abaixo usa uma amostra do conjunto de teste para manter o tempo de execução controlado.

Interpretação geral:

- valores SHAP positivos aumentam a probabilidade estimada de compra premium;
- valores SHAP negativos reduzem a probabilidade estimada de compra premium;
- a magnitude indica força da contribuição.

In [ ]:
import shap

sample_size = min(2_000, len(X_test))
background_size = min(500, len(X_train))

X_train_sample = X_train.sample(background_size, random_state=RANDOM_STATE)
X_test_sample = X_test.sample(sample_size, random_state=RANDOM_STATE)

preprocess_fitted = best_pipeline.named_steps["preprocess"]
model_fitted = best_pipeline.named_steps["model"]

X_train_transformed = preprocess_fitted.transform(X_train_sample)
X_test_transformed = preprocess_fitted.transform(X_test_sample)

if not isinstance(X_train_transformed, np.ndarray):
    X_train_transformed = X_train_transformed.toarray()
if not isinstance(X_test_transformed, np.ndarray):
    X_test_transformed = X_test_transformed.toarray()

X_train_transformed_df = pd.DataFrame(X_train_transformed, columns=transformed_feature_names, index=X_train_sample.index)
X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=transformed_feature_names, index=X_test_sample.index)

if isinstance(model_fitted, LogisticRegression):
    explainer = shap.LinearExplainer(model_fitted, X_train_transformed_df)
elif isinstance(model_fitted, (RandomForestClassifier, HistGradientBoostingClassifier)):
    explainer = shap.TreeExplainer(model_fitted)
else:
    explainer = shap.Explainer(model_fitted.predict_proba, X_train_transformed_df)

shap_values = explainer(X_test_transformed_df)
shap_values

In [ ]:
def positive_class_shap_explanation(explanation: shap.Explanation) -> shap.Explanation:
    """Normaliza a saída do SHAP para explicar a classe premium."""
    values = explanation.values
    base_values = explanation.base_values
    data = explanation.data

    if values.ndim == 3:
        # Formato comum em classificadores: amostras x features x classes.
        if values.shape[-1] == 2:
            values = values[:, :, 1]
            if np.asarray(base_values).ndim == 2:
                base_values = base_values[:, 1]
        # Fallback para formato amostras x classes x features.
        elif values.shape[1] == 2:
            values = values[:, 1, :]
            if np.asarray(base_values).ndim == 2:
                base_values = base_values[:, 1]

    return shap.Explanation(
        values=values,
        base_values=base_values,
        data=data,
        feature_names=explanation.feature_names,
    )

shap_positive = positive_class_shap_explanation(shap_values)
print(f"Formato dos valores SHAP da classe premium: {shap_positive.values.shape}")

In [ ]:
shap.summary_plot(
    shap_positive.values,
    X_test_transformed_df,
    feature_names=transformed_feature_names,
    plot_type="bar",
    max_display=20,
    show=False,
)
plt.title("SHAP Feature Importance - classe premium")
plt.tight_layout()
plt.show()

shap.summary_plot(
    shap_positive.values,
    X_test_transformed_df,
    feature_names=transformed_feature_names,
    max_display=20,
    show=False,
)
plt.title("SHAP Beeswarm - impacto e direção das variáveis")
plt.tight_layout()
plt.show()

In [ ]:
def map_transformed_to_original_feature(transformed_feature: str) -> str:
    if transformed_feature in numeric_features:
        return transformed_feature

    matching_categorical = [
        feature for feature in categorical_features if transformed_feature.startswith(f"{feature}_")
    ]
    if matching_categorical:
        return max(matching_categorical, key=len)

    return transformed_feature

shap_importance_original_df = (
    pd.DataFrame(
        {
            "transformed_feature": transformed_feature_names,
            "original_feature": [map_transformed_to_original_feature(feature) for feature in transformed_feature_names],
            "mean_abs_shap": np.abs(shap_positive.values).mean(axis=0),
        }
    )
    .groupby("original_feature", as_index=False)["mean_abs_shap"]
    .sum()
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

display(shap_importance_original_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=shap_importance_original_df.head(15), x="mean_abs_shap", y="original_feature")
plt.title("Importância SHAP agregada por variável original")
plt.xlabel("Média do valor absoluto SHAP")
plt.ylabel("Variável original")
plt.show()

In [ ]:
sample_predictions = best_pipeline.predict_proba(X_test_sample)[:, 1]
example_position = int(np.argmax(sample_predictions))
example_index = X_test_sample.index[example_position]

print(f"Exemplo local escolhido: índice {example_index}")
print(f"Probabilidade prevista de compra premium: {sample_predictions[example_position]:.2%}")
display(X_test_sample.loc[[example_index]])

shap.plots.waterfall(shap_positive[example_position], max_display=15)

## 9. Síntese interpretável dos fatores

As tabelas abaixo organizam os resultados de XAI em duas leituras:

1. **Variáveis originais mais importantes**: melhor para responder quais dimensões influenciam a probabilidade de compra premium.
2. **Categorias/variáveis transformadas com maior impacto SHAP**: melhor para entender quais valores específicos empurram a previsão para cima ou para baixo.

Atenção: o modelo identifica associações preditivas no dataset. Para afirmar causalidade, seria necessário desenho experimental ou quase experimental.

In [ ]:
shap_transformed_summary_df = (
    pd.DataFrame(
        {
            "feature": transformed_feature_names,
            "original_feature": [map_transformed_to_original_feature(feature) for feature in transformed_feature_names],
            "mean_abs_shap": np.abs(shap_positive.values).mean(axis=0),
            "mean_shap": shap_positive.values.mean(axis=0),
            "share_positive_impact": (shap_positive.values > 0).mean(axis=0),
        }
    )
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

shap_transformed_summary_df["typical_direction"] = np.where(
    shap_transformed_summary_df["mean_shap"] > 0,
    "aumenta probabilidade premium",
    "reduz probabilidade premium",
)

print("Top variáveis originais por impacto SHAP:")
display(shap_importance_original_df.head(10))

print("Top variáveis/categorias transformadas por impacto SHAP:")
display(shap_transformed_summary_df.head(20))

In [ ]:
top_original_features = shap_importance_original_df.head(5)["original_feature"].tolist()
top_transformed_rows = shap_transformed_summary_df.head(8)[
    ["feature", "original_feature", "typical_direction", "mean_abs_shap", "share_positive_impact"]
]

print("Resposta orientada pela modelagem:")
print(
    "Os fatores que mais influenciam a probabilidade de uma compra premium, segundo o modelo e o SHAP, são: "
    + ", ".join(top_original_features)
    + "."
)
print("\nDetalhamento das categorias/variáveis de maior impacto:")
display(top_transformed_rows)

print("\nComo interpretar:")
print("- Use a importância por permutação para priorizar dimensões explicativas no nível das variáveis originais.")
print("- Use o SHAP beeswarm para observar a direção do efeito e a dispersão entre clientes/transações.")
print("- Use o waterfall para explicar uma previsão individual de compra premium.")

## 10. Conclusões esperadas

Ao executar o notebook, a resposta final deve ser baseada nas tabelas e gráficos gerados. Em datasets de varejo como este, é comum que categorias de produto tenham forte influência, seguidas por características demográficas e contexto de cidade/permanência.

A conclusão deve seguir este formato:

- **Fatores mais influentes**: variáveis no topo de `shap_importance_original_df` e `permutation_importance_df`.
- **Direção dos efeitos**: categorias ou faixas com `mean_shap` positivo aumentam a probabilidade estimada de compra premium; valores negativos reduzem.
- **Confiança do modelo**: considerar ROC AUC, Average Precision e matriz de confusão antes de usar os achados para decisão.
- **Limitação principal**: os resultados são preditivos e explicativos do modelo, não uma prova causal de que determinado fator provoca compras premium.

## Próximos passos possíveis

- Testar definições alternativas de compra premium, como percentil 90 ou ticket acima de um valor de negócio.
- Agregar dados por cliente para prever se um cliente terá ao menos uma compra premium, em vez de classificar transações individualmente.
- Comparar com modelos calibrados de probabilidade caso o objetivo seja segmentação comercial.